In [ ]:
%%capture
import os
from pathlib import Path
import pandas as pd
import numpy as np

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
reports_folder = Path(os.environ["INTECOMM_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)

In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858, get_patientlog_df
from edc_pdutils.dataframes import get_crf, get_subject_visit
from intecomm_analytics.dataframes.get_df_main_1858 import get_location_update

In [ ]:
df_main = get_df_main_1858(None)

In [ ]:
df_location_update = get_location_update(df_main)
df_visit = get_subject_visit("intecomm_subject.subjectvisit")
df_unwell = df_visit[df_visit.reason_unscheduled=="patient_unwell_outpatient"].sort_values(by=["subject_identifier"]).reset_index(drop=True)
df_unwell = df_unwell.merge(df_location_update[["subject_identifier", "visit_code", "direction", 'comments']], how="left", on=["subject_identifier", "visit_code"])[["subject_identifier", "visit_code", "reason_unscheduled", "direction", "comments"]]
df_unwell = df_unwell.merge(df_main[["subject_identifier", "assignment", "pp"]], on=["subject_identifier"], how="left")
df_unwell["comments"] = df_unwell["comments"].apply(lambda x: np.nan if x=="" else x )

In [ ]:
df_unwell.to_csv(analysis_folder / "unwell.csv", index=False)

In [ ]:
variable_labels = {
    "subject_identifier": "subject/participant unique identifier",
    "visit_code": "Float of visit code + viist_code_sequence ",
    "reason_unscheduled": "Reason for unscheduled visit",
    "direction": "a->b is community patient attended at the facility (LocationUpdate)",
    "assignment": "Intention-to-treat a=comm, b=facility",
    "pp": "Per protocol assignment a=comm, b=facility",
    "comments": "reason from LocationUpdate CRF, not required."
}
df_unwell.to_stata(
    path=analysis_folder / "df_unwell.dta",
    variable_labels=variable_labels,
    version=118,
    write_index=False,
)